# HPD 2 — Ejercicios evaluables

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd2-evaluables.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 % (parte del 30 % de entregas prácticas)
>
> **Plazo:** 7 días tras la sesión presencial.
>
> **Requisito previo:** haber completado el notebook de la sesión
> presencial (`bloque2/hpd2-agente-tool-calling.qmd`).
>
> **Entrega:** Notebook `.ipynb` ejecutado con todas las celdas
> completas. Cada ejercicio especifica qué variable debe contener el
> resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto (`eval1_resultado`, `eval2_tabla`, `eval3_respuesta`,
> `eval4_log`). El script de corrección ejecutará tu notebook e
> inspeccionará esas variables. **Si la variable no existe o tiene un
> tipo incorrecto, el ejercicio se puntúa como 0.**
>
> Para autoevaluarte antes de entregar:
>
> ``` bash
> python scripts/corregir_hpd2.py tu_notebook.ipynb
> ```

In [1]:
!pip install -q langchain langchain-openai langchain-community pandas matplotlib seaborn python-dotenv

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io, base64

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

plt.rcParams["figure.dpi"] = 100

LLM_KEY = os.getenv("LLM_API_KEY")
LLM_URL = "https://llamus.cs.us.es/api/v1"

if not LLM_KEY:
    print("⚠️  Crea .env con LLM_API_KEY=tu_key. Las evaluaciones con LLM no se ejecutarán.")
else:
    llm = ChatOpenAI(model="gemma4:e2b-mlx", base_url=LLM_URL, api_key=LLM_KEY, temperature=0)
    print("✅ LLM configurado.")

df = sns.load_dataset("titanic")
print(f"Dataset Titanic: {df.shape[0]} filas × {df.shape[1]} columnas")

✅ LLM configurado.
Dataset Titanic: 891 filas × 15 columnas

In [3]:
# ─── Herramienta base compartida para todos los ejercicios ───
@tool
def consulta_pandas(expresion: str) -> str:
    """
    Ejecuta una expresión de pandas sobre el DataFrame df del Titanic.
    El df tiene columnas: survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone.
    
    Args:
        expresion: expresión Python usando 'df' (ej: "df.groupby('pclass')['fare'].mean()")
    
    Returns:
        Resultado de la consulta como string.
    """
    try:
        resultado = eval(expresion, {"df": df, "pd": pd, "__builtins__": {}})
        return str(resultado)
    except Exception as e:
        return f"Error: {e}"

@tool
def histograma(columna: str, titulo: str = "") -> str:
    """Genera un histograma de una columna numérica del Titanic (Age, Fare...)."""
    df[columna].hist()
    plt.title(titulo or f"Histograma de {columna}")
    return f"Histograma de {columna} generado"

@tool
def barras(columna: str, agrupar_por: str = "") -> str:
    """Genera un gráfico de barras. agrupar_por: opcional (Sex, Pclass...)."""
    if agrupar_por:
        df.groupby(agrupar_por)[columna].mean().plot.bar()
    else:
        df[columna].value_counts().plot.bar()
    plt.title(f"Barras de {columna}")
    return f"Barras de {columna} generado"

@tool
def boxplot(columna: str, agrupar_por: str = "") -> str:
    """Genera un boxplot. agrupar_por: opcional para comparar categorías."""
    if agrupar_por:
        df.boxplot(columna, by=agrupar_por)
    else:
        df.boxplot(columna)
    return f"Boxplot de {columna} generado"

@tool
def codigo_libre(codigo: str) -> str:
    """
    Ejecuta código matplotlib libre sobre df (Titanic).
    Úsalo para gráficos que ninguna otra tool cubra.

    REGLAS: usa plt.<funcion>(), NUNCA df.plot().
    NO uses plt.show() (se elimina automáticamente).
    NO leas archivos (df ya cargado).
    Variables: df, plt, pd, np.

    Ejemplo: plt.scatter(df['Age'], df['Fare'], c=df['Survived'])
    """
    codigo = codigo.replace("plt.show()", "")
    try:
        plt.figure()
        exec(codigo, {"plt": plt, "pd": pd, "df": df, "np": np})
        buf = io.BytesIO()
        plt.savefig(buf, format="png", bbox_inches="tight")
        plt.close()
        return f"Gráfico generado ({len(buf.getvalue())} bytes)"
    except Exception as e:
        plt.close()
        return f"Error: {e}. Reintenta con plt.<funcion>() en vez de df.plot()"

SYSTEM_PROMPT = "Eres un analista de datos del Titanic. Usa las herramientas SIEMPRE que la pregunta requiera datos del Titanic. Responde en español, de forma concisa."

print("Herramientas y system prompt base listos.")

Herramientas y system prompt base listos.

------------------------------------------------------------------------

## Ejercicio 1 — Añade una herramienta de correlación (2.5 puntos)

Implementa una tercera herramienta `correlacion(col1: str, col2: str)`
que calcule la correlación de Pearson entre dos columnas numéricas del
DataFrame. La herramienta debe manejar columnas no numéricas o
inexistentes con un mensaje de error descriptivo. Después, ejecuta el
agente con las 3 herramientas y haz una pregunta que active
específicamente esta nueva tool.

| Criterio | Puntos |
|------------------------------------|------------------------------------|
| Herramienta `correlacion` implementada con docstring completo | 1.0 |
| Maneja errores (columna no numérica, inexistente, valores nulos) | 0.5 |
| `eval1_resultado` contiene la respuesta del agente a una pregunta que usa `correlacion` | 0.5 |
| La respuesta es coherente con los datos | 0.5 |

In [4]:
# ─── Implementa aquí la herramienta correlacion ───
@tool
def correlacion(col1: str, col2: str) -> str:
    """
    Calcula la correlación de Pearson entre dos columnas numéricas del DataFrame df.
    
    Args:
        col1: nombre de la primera columna
        col2: nombre de la segunda columna
    
    Returns:
        Coeficiente de correlación como string.
    """
    # TODO: Implementar
    # - Validar que col1 y col2 existen en df.columns
    # - Validar que ambas son numéricas  
    # - Calcular correlación con .corr()
    # - Manejar errores
    pass

# ─── Crea y ejecuta el agente con 3 herramientas ───
# tools = [consulta_pandas, histograma, barras, boxplot, codigo_libre, correlacion]
# agente = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)
# result = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})

# ─── Pregunta que active la tool correlacion ───
# pregunta_1 = "¿Existe correlación entre la edad y la tarifa pagada?"

# ─── Resultado esperado por el corrector ───
# eval1_resultado debe ser un string con la respuesta final del agente

eval1_resultado = None  # ← asigna aquí la respuesta del agente (str)

In [5]:
# ─── Auto-verificación ───
assert isinstance(eval1_resultado, str), "❌ eval1_resultado debe ser un string"
assert len(eval1_resultado) > 10, f"❌ Respuesta demasiado corta ({len(eval1_resultado)} chars)"
print("✅ Ejercicio 1: formato correcto")

------------------------------------------------------------------------

## Ejercicio 2 — Precisión del agente (2.5 puntos)

Ejecuta el agente base (con `consulta_pandas` + tools de visualización)
sobre las siguientes 5 preguntas nuevas. Para cada una, evalúa si la
respuesta es correcta (escala 0-5) y anota qué herramienta usó. Si no
usó ninguna herramienta cuando debería, puntúa 0.

| Criterio                                       | Puntos |
|------------------------------------------------|--------|
| 5 preguntas ejecutadas con el agente           | 1.0    |
| Tabla completada con tool usada y puntuación   | 0.5    |
| Al menos 3/5 preguntas obtienen puntuación ≥ 3 | 1.0    |

In [6]:
# ─── Preguntas de evaluación ───
preguntas_eval = [
    "¿Cuál es la tarifa media pagada por los pasajeros de primera clase?",
    "¿Cuántos pasajeros viajaban solos (alone=True)?",
    "Genera un gráfico de barras que muestre la supervivencia por clase (pclass)",
    "¿Cuál es la edad del pasajero más joven y del más mayor?",
    "¿Hay más supervivientes entre los que viajaban con familia (sibsp > 0) que entre los que viajaban solos (sibsp == 0)?"
]

# ─── Crea el agente base ───
# tools = [consulta_pandas, histograma, barras, boxplot, codigo_libre]
# agente = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)
# result = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})

# Para cada pregunta, ejecuta executor.invoke({"input": pregunta})
# Evalúa la respuesta y completa eval2_tabla

# ─── Resultado esperado por el corrector ───
# eval2_tabla debe ser una lista de dicts con:
# [{"pregunta": str, "tool_usada": str, "puntuacion": int, "observaciones": str}, ...]

eval2_tabla = []  # ← lista de 5 dicts

In [7]:
# ─── Auto-verificación ───
assert isinstance(eval2_tabla, list), "❌ eval2_tabla debe ser una lista"
assert len(eval2_tabla) == 5, f"❌ Se esperaban 5 evaluaciones, tienes {len(eval2_tabla)}"
for i, row in enumerate(eval2_tabla):
    for k in ("pregunta", "tool_usada", "puntuacion", "observaciones"):
        assert k in row, f"❌ Falta clave '{k}' en fila {i}"
    assert isinstance(row["puntuacion"], int), f"❌ puntuacion debe ser int en fila {i}"
    assert 0 <= row["puntuacion"] <= 5, f"❌ puntuacion fuera de rango en fila {i}"
print("✅ Ejercicio 2: formato correcto")

------------------------------------------------------------------------

## Ejercicio 3 — Detección de preguntas fuera de ámbito (2.5 puntos)

Modifica el `SYSTEM_PROMPT` para que el agente solo responda preguntas
relacionadas con el dataset del Titanic. Si la pregunta es sobre otro
tema, debe responder *“No tengo datos sobre eso. Solo puedo analizar el
dataset del Titanic.”* **sin llamar a ninguna herramienta**.

Prueba con 2 preguntas válidas y 2 preguntas fuera de ámbito.

| Criterio                                                     | Puntos |
|--------------------------------------------------------------|--------|
| System prompt modificado correctamente                       | 1.0    |
| 2 preguntas válidas respondidas con herramientas             | 0.75   |
| 2 preguntas fuera de ámbito rechazadas sin usar herramientas | 0.75   |

In [8]:
# ─── Define el nuevo system prompt ───
# SYSTEM_PROMPT_FUERA_AMBITO = """..."""

# ─── Crea el agente con el nuevo prompt ───

# ─── Preguntas de prueba ───
# validas = ["¿Cuántos hombres sobrevivieron?", "Muestra la distribución de edades con un gráfico"]
# fuera_ambito = ["¿Quién ganó el Mundial de fútbol en 2022?", "Escribe un poema sobre el mar"]

# ─── Resultado esperado por el corrector ───
# eval3_respuesta debe ser un dict:
# {"valida_1": str, "valida_2": str, "fuera_1": str, "fuera_2": str}
# Donde fuera_1 y fuera_2 contienen "No tengo datos" o similar

eval3_respuesta = {}  # ← dict con 4 respuestas

In [9]:
# ─── Auto-verificación ───
assert isinstance(eval3_respuesta, dict), "❌ eval3_respuesta debe ser un dict"
for k in ("valida_1", "valida_2", "fuera_1", "fuera_2"):
    assert k in eval3_respuesta, f"❌ Falta clave '{k}'"
    assert isinstance(eval3_respuesta[k], str), f"❌ {k} debe ser str"
    assert len(eval3_respuesta[k]) > 5, f"❌ {k} demasiado corta"
# Verificar que fuera_1 y fuera_2 rechazan
for k in ("fuera_1", "fuera_2"):
    assert "titanic" in eval3_respuesta[k].lower() or "no tengo" in eval3_respuesta[k].lower() or "datos" in eval3_respuesta[k].lower(), \
        f"❌ {k} debería rechazar la pregunta pero respondió: {eval3_respuesta[k][:80]}"
print("✅ Ejercicio 3: formato correcto")

------------------------------------------------------------------------

## Ejercicio 4 — Ground truth propio (2.5 puntos)

Escribe **3 preguntas originales** (que no estén en los ejercicios
anteriores ni en el notebook de clase) y define manualmente la respuesta
que esperas. Ejecuta el agente y compara la respuesta obtenida con tu
ground truth. Evalúa cada una como correcta/parcialmente
correcta/incorrecta.

| Criterio                                                | Puntos |
|---------------------------------------------------------|--------|
| 3 preguntas originales definidas con respuesta esperada | 1.0    |
| Agente ejecutado sobre las 3 preguntas                  | 1.0    |
| Evaluación de coincidencia con ground truth             | 0.5    |

In [10]:
# ─── Define tus 3 preguntas con ground truth ───
mi_ground_truth = {
    # "¿tu pregunta original 1?": "respuesta esperada (aproximada)",
    # "¿tu pregunta original 2?": "respuesta esperada (aproximada)",
    # "¿tu pregunta original 3?": "respuesta esperada (aproximada)",
}

# ─── Ejecuta el agente para cada pregunta ───
# Para cada pregunta, guarda en eval4_log un dict con:
# { "pregunta": str, "esperado": str, "obtenido": str, "evaluacion": "correcta"|"parcial"|"incorrecta" }

# ─── Resultado esperado por el corrector ───
# eval4_log debe ser una lista de 3 dicts

eval4_log = []  # ← lista de 3 dicts

In [11]:
# ─── Auto-verificación ───
assert isinstance(eval4_log, list), "❌ eval4_log debe ser una lista"
assert len(eval4_log) == 3, f"❌ Se esperaban 3 entradas, tienes {len(eval4_log)}"
for i, row in enumerate(eval4_log):
    for k in ("pregunta", "esperado", "obtenido", "evaluacion"):
        assert k in row, f"❌ Falta clave '{k}' en fila {i}"
    assert row["evaluacion"] in ("correcta", "parcial", "incorrecta"), \
        f"❌ evaluacion inválida en fila {i}: {row['evaluacion']}"
print("✅ Ejercicio 4: formato correcto")